# ABSA clean ablation training — fresh run

Notebook này dùng package `absa_clean_ablation_fresh.zip` và **train lại từ đầu**.

Nó không dùng checkpoint/log/history cũ. Ở phần setup, notebook sẽ xóa các thư mục output trong `/kaggle/working` trước khi chạy:

- `tuning_ablation_core`
- `outputs_ablation_core`
- `tuning_ablation_desc`
- `outputs_ablation_desc`
- project folder đã extract trước đó

Bật **GPU** và **Internet** trong Kaggle để tải pretrained model lần đầu.


In [ ]:
from pathlib import Path
import os, shutil, zipfile, json, subprocess, sys

print("Python:", sys.version)
print("Kaggle working exists:", Path("/kaggle/working").exists())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found; GPU may be disabled.")


## 1. Extract project sạch vào `/kaggle/working`

Cell này xóa project/output cũ trong `/kaggle/working`, sau đó extract lại từ zip trong `/kaggle/input`.


In [ ]:
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT = Path('/kaggle/input')
PROJECT = WORK / 'absa_clean_ablation_fresh'

CLEAN_RUN = True
if CLEAN_RUN:
    for path in [
        PROJECT,
        WORK / 'tuning_ablation_core',
        WORK / 'outputs_ablation_core',
        WORK / 'tuning_ablation_desc',
        WORK / 'outputs_ablation_desc',
        WORK / 'absa_ablation_results_fresh.zip',
    ]:
        if path.exists():
            print('Removing old:', path)
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()

zip_candidates = []
if INPUT.exists():
    zip_candidates = list(INPUT.rglob('absa_clean_ablation_fresh.zip'))
    if not zip_candidates:
        zip_candidates = list(INPUT.rglob('absa_clean_ablation*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK)
else:
    train_candidates = list(INPUT.rglob('train.py')) if INPUT.exists() else []
    if not train_candidates:
        raise FileNotFoundError('Không tìm thấy absa_clean_ablation_fresh.zip hoặc train.py trong /kaggle/input')
    src = train_candidates[0].parent
    print('Copying source folder:', src)
    shutil.copytree(src, PROJECT, dirs_exist_ok=True)

if not (PROJECT / 'train.py').exists():
    candidates = [p.parent for p in WORK.rglob('train.py') if 'absa_clean_ablation' in str(p.parent)]
    if not candidates:
        raise FileNotFoundError('Extract xong nhưng không tìm thấy train.py')
    PROJECT = candidates[0]

print('PROJECT =', PROJECT)
print('Files:', sorted([p.name for p in PROJECT.iterdir()])[:30])


## 2. Cài requirements


In [ ]:
os.chdir(PROJECT)
print('cwd =', Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 3. Kiểm tra package không có output/checkpoint cũ


In [ ]:
os.chdir(PROJECT)
forbidden_markers = ['tuning_', 'outputs_', 'wandb', 'runs', 'logs', '__pycache__']
forbidden_suffixes = {'.pt', '.pth', '.bin', '.safetensors'}
found = []
for p in PROJECT.rglob('*'):
    rel = str(p.relative_to(PROJECT))
    if any(part in rel for part in forbidden_markers) or p.suffix in forbidden_suffixes:
        found.append(rel)

print('Forbidden/history-like files found:', len(found))
for item in found[:50]:
    print('-', item)
assert not found, 'Package không sạch: có file output/checkpoint/history-like.'

print('\nDataset files:')
for p in sorted(Path('dataset/rest').glob('*')):
    print('-', p, p.stat().st_size)

print('\nSearch-space help:')
subprocess.run([sys.executable, 'tune.py', '--help'], check=False)


## 4. Chạy core ablation từ đầu

Core ablation không dùng description để tránh confound.

Các trial chính:

- `baseline_cls`
- `aspect_attention_term`
- `hybrid_term`
- `cls_neutral125`
- `cls_neutral150`
- `aspect_attention_neutral125`

Nếu GPU OOM, giảm `BATCH_SIZE=4` và tăng `GRAD_ACCUM=4`.


In [ ]:
os.chdir(PROJECT)

VALID_RATIO = 0.15
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 12
FINAL_EPOCHS = 15
SEED_PAIRS = '2024:2024,42:42,3407:3407'
FINAL_SEEDS = '2024,42,3407'

cmd = [
    sys.executable, 'tune.py',
    '--dataset', 'rest',
    '--search-space', 'ablation_core',
    '--validation-seed-pairs', SEED_PAIRS,
    '--valid-ratio', str(VALID_RATIO),
    '--final-seeds', FINAL_SEEDS,
    '--epochs', str(EPOCHS),
    '--final-epochs', str(FINAL_EPOCHS),
    '--patience', '3',
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--amp',
    '--gradient-checkpointing',
    '--run-final',
    '--output-dir', str(WORK / 'tuning_ablation_core'),
    '--final-output-dir', str(WORK / 'outputs_ablation_core'),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 5. Xem nhanh kết quả core


In [ ]:
import pandas as pd

summary_path = WORK / 'tuning_ablation_core' / 'tuning_summary.csv'
final_path = WORK / 'outputs_ablation_core' / 'final_runs_summary.csv'
meanstd_path = WORK / 'outputs_ablation_core' / 'final_metrics_mean_std.json'

print('Tuning summary:', summary_path)
summary = pd.read_csv(summary_path)
cols = [c for c in ['trial','mean_macro_f1','std_macro_f1','robust_macro_f1','mean_accuracy','mean_ece','pooling_strategy','input_format','neutral_weight_multiplier'] if c in summary.columns]
display(summary[cols])

if final_path.exists():
    print('Final runs:', final_path)
    final = pd.read_csv(final_path)
    display(final)

if meanstd_path.exists():
    print(json.dumps(json.loads(meanstd_path.read_text()), indent=2))


## 6. Optional: chạy description ablation riêng

Chỉ bật cell này sau khi core ablation xong. Mục đích là kiểm tra riêng `aspects.json`, không gộp với neutral weight.


In [ ]:
RUN_DESC_ABLATION = False  # đổi thành True nếu muốn chạy thêm aspect description ablation

if RUN_DESC_ABLATION:
    desc_cmd = [
        sys.executable, 'tune.py',
        '--dataset', 'rest',
        '--search-space', 'ablation_desc',
        '--validation-seed-pairs', SEED_PAIRS,
        '--valid-ratio', str(VALID_RATIO),
        '--final-seeds', FINAL_SEEDS,
        '--epochs', str(EPOCHS),
        '--final-epochs', str(FINAL_EPOCHS),
        '--patience', '3',
        '--batch-size', str(BATCH_SIZE),
        '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--amp',
        '--gradient-checkpointing',
        '--run-final',
        '--output-dir', str(WORK / 'tuning_ablation_desc'),
        '--final-output-dir', str(WORK / 'outputs_ablation_desc'),
    ]
    print(' '.join(desc_cmd))
    subprocess.run(desc_cmd, check=True)
else:
    print('Skipped. Set RUN_DESC_ABLATION = True để chạy.')


## 7. Zip kết quả để tải về

Mặc định không zip checkpoint `.pt` để file nhẹ hơn. Nếu muốn lưu checkpoint, đổi `INCLUDE_CHECKPOINTS=True`.


In [ ]:
INCLUDE_CHECKPOINTS = False
zip_out = WORK / 'absa_ablation_results_fresh.zip'
folders = [WORK / 'tuning_ablation_core', WORK / 'outputs_ablation_core', WORK / 'tuning_ablation_desc', WORK / 'outputs_ablation_desc']

with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        if not folder.exists():
            continue
        for path in folder.rglob('*'):
            if path.is_file():
                if not INCLUDE_CHECKPOINTS and path.suffix in {'.pt', '.pth', '.bin', '.safetensors'}:
                    continue
                zf.write(path, path.relative_to(WORK))

print('Created:', zip_out)
print('Size MB:', round(zip_out.stat().st_size / 1024 / 1024, 2))
